# 🎮 ĐỀ TÀI: NGHIÊN CỨU & ỨNG DỤNG GENERATIVE AI TRONG SẢN XUẤT GAME
## Pipeline Tự Động Tạo Hình Nhân Vật: Từ Ý Tưởng Văn Bản (Text) $\rightarrow$ Concept Art 2D $\rightarrow$ Mô Hình 3D Hoàn Chỉnh
---
### 📌 1. Bối cảnh & Mục tiêu đề tài bài báo:
- **Vấn đề thực tế trong ngành Game:** Quy trình thiết kế nhân vật truyền thống (Concept Artist vẽ 2D $\rightarrow$ 3D Modeler nặn khối $\rightarrow$ Retopology $\rightarrow$ UV Unwrapping $\rightarrow$ Texturing) thường tiêu tốn từ **3 đến 7 ngày công** cho một nhân vật cơ bản.
- **Giải pháp AI (Two-Stage Generative Pipeline):**
  1. **Giai đoạn 1 (Text-to-2D):** Sử dụng mô hình **Diffusion (Stable Diffusion / SDXL)** với prompt chuyên dụng để tạo hình Concept Art nhân vật game ở tư thế chuẩn (T-pose / Front-view, tách biệt nền).
  2. **Giai đoạn 2 (2D-to-3D):** Sử dụng mô hình tái tạo không gian 3D **TripoSR (Transformer-based Feedforward 3D Reconstruction)** để tạo lưới đa giác (Polygon Mesh) và vân bề mặt (Texture) trong vòng **dưới 2 giây**.
- **Kết quả đầu ra:** Bộ số liệu thực nghiệm (thời gian xử lý, số lượng đa giác, VRAM, so sánh hiệu năng) và file 3D (`.glb`) tương thích ngay với **Unity / Unreal Engine / Blender**.


In [ ]:
# =============================================================================
# BƯỚC 1: Cài đặt môi trường AI (Diffusers, Stable Diffusion, TripoSR, PyTorch GPU)
# =============================================================================
!nvidia-smi
print('⏳ Đang cài đặt các thư viện Deep Learning... (mất khoảng 1.5 - 2 phút)')

# 1. Cài đặt các thư viện xử lý ảnh 2D & Diffusion Models đồng bộ mới nhất
!pip install -q -U transformers diffusers huggingface-hub accelerate safetensors

# 2. Cài đặt các thư viện tái tạo 3D & Marching Cubes (dùng PyMCubes không cần build)
!pip install -q PyMCubes trimesh pyvista gradio
!pip install -q "rembg[gpu]" onnxruntime-gpu einops omegaconf

# 3. Tải mã nguồn TripoSR
![ ! -d "/content/TripoSR" ] && git clone https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR

print('\n✅ CÀI ĐẶT MÔI TRƯỜNG THÀNH CÔNG!')


In [ ]:
# =============================================================================
# BƯỚC 2: Khởi tạo mô hình 2D Concept Generator (Stable Diffusion)
# =============================================================================
import subprocess, sys
try:
    import diffusers
except Exception:
    print('⚡ Đang tự động đồng bộ transformers & diffusers mới nhất...')
    subprocess.check_call(['pip', 'install', '-q', '-U', 'transformers', 'diffusers', 'huggingface-hub'])

import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ Đang tải mô hình tạo ảnh 2D Concept Art trên {device}...")

model_id = "runwayml/stable-diffusion-v1-5"
pipe_2d = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    safety_checker=None
)
pipe_2d.scheduler = DPMSolverMultistepScheduler.from_config(pipe_2d.scheduler.config)
pipe_2d.to(device)
print("✅ Mô hình tạo Concept Art 2D đã sẵn sàng!")

def generate_2d_character(prompt_desc, negative_prompt="", steps=25, guidance=7.5):
    """
    Hàm sinh concept art nhân vật game tối ưu cho dựng 3D (A-pose, nền trắng, toàn thân)
    Tự động chuẩn hóa để không vượt quá 77 tokens của CLIP.
    """
    words = prompt_desc.strip().split()
    if len(words) > 40:
        clean_desc = " ".join(words[:40])
        print(f"ℹ️ Đã tự động tối ưu prompt từ {len(words)} từ xuống 40 từ để tương thích CLIP 77 tokens.")
    else:
        clean_desc = prompt_desc.strip()
        
    full_prompt = (
        f"full body game character concept art of {clean_desc}, "
        "front view, centered, standing pose, neutral A-pose, white solid background, "
        "game asset design, high quality, 8k resolution, clean edges, flat lighting"
    )
    neg_prompt = (
        "cropped, cut off, bad anatomy, deformed limbs, blurry, noise, "
        "complex background, shadows on background, dark background, side view, back view, text, watermark "
        + negative_prompt
    )
    
    image = pipe_2d(
        prompt=full_prompt,
        negative_prompt=neg_prompt,
        num_inference_steps=steps,
        guidance_scale=guidance,
        height=512,
        width=512
    ).images[0]
    return image


In [ ]:
# =============================================================================
# BƯỚC 3: Khởi tạo mô hình 2D-to-3D Reconstruction (TripoSR Auto-Orient & Enhance)
# =============================================================================
import sys, os, time, types, subprocess, trimesh, numpy as np
sys.path.append('/content/TripoSR')

try:
    from PIL import Image
    import PIL._typing
except Exception:
    print('⚡ Đang tự động sửa lỗi thư viện Pillow...')
    subprocess.check_call(['pip', 'install', '-q', '--upgrade', '--force-reinstall', 'pillow'])
    import importlib, PIL
    importlib.reload(PIL)

import types, mcubes
fake_mc = types.ModuleType('torchmcubes')
def custom_mc(density, isovalue=0.0):
    d_np = density.detach().cpu().numpy()
    verts, tris = mcubes.marching_cubes(d_np, isovalue)
    return torch.from_numpy(verts).float().to(density.device), torch.from_numpy(tris.astype('int64')).to(density.device)
fake_mc.marching_cubes = custom_mc
sys.modules['torchmcubes'] = fake_mc

from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground
import rembg

print(f"⚡ Đang tải mô hình TripoSR vào {device}...")
from huggingface_hub import hf_hub_download
from omegaconf import OmegaConf

try:
    model_3d = TSR.from_pretrained("stabilityai/TripoSR", config_name="config.yaml", weight_name="model.ckpt")
except RuntimeError as e:
    if 'image_tokenizer' in str(e):
        print('⚡ Đang tự động tương thích hóa trọng số cho Transformers mới...')
        cfg_path = hf_hub_download(repo_id='stabilityai/TripoSR', filename='config.yaml')
        weight_path = hf_hub_download(repo_id='stabilityai/TripoSR', filename='model.ckpt')
        cfg = OmegaConf.load(cfg_path)
        OmegaConf.resolve(cfg)
        model_3d = TSR(cfg)
        state_dict = torch.load(weight_path, map_location='cpu')
        new_state_dict = {}
        for k, v in state_dict.items():
            new_k = k
            if 'image_tokenizer.model.encoder.layer.' in k:
                new_k = new_k.replace('encoder.layer.', 'layers.')
                new_k = new_k.replace('attention.attention.query.', 'attention.q_proj.')
                new_k = new_k.replace('attention.attention.key.', 'attention.k_proj.')
                new_k = new_k.replace('attention.attention.value.', 'attention.v_proj.')
                new_k = new_k.replace('attention.output.dense.', 'attention.o_proj.')
                new_k = new_k.replace('intermediate.dense.', 'mlp.fc1.')
                new_k = new_k.replace('output.dense.', 'mlp.fc2.')
            new_state_dict[new_k] = v
        model_3d.load_state_dict(new_state_dict, strict=False)
    else:
        raise e

model_3d.renderer.set_chunk_size(131072)
model_3d.to(device)
rembg_session = rembg.new_session('u2net')
print("✅ Mô hình TripoSR (2D-to-3D) đã sẵn sàng!")

os.makedirs("/content/export_3d", exist_ok=True)

def orient_mesh_upright(mesh):
    """
    Chuẩn hóa góc quay nhân vật 3D đứng thẳng 100%:
    - Đưa trục Chiều cao (Đầu - Chân) thẳng đứng theo trục Y (+Y là Đầu, -Y là Chân)
    - Đưa Mặt trước (Ngực/Mặt) nhìn thẳng vào camera (+Z)
    """
    import numpy as np, trimesh
    extents = np.ptp(mesh.vertices, axis=0)
    h_axis = np.argmax(extents)
    
    # Nếu chiều cao đang nằm dọc theo trục X (mặc định sau Marching Cubes)
    if h_axis == 0:
        mesh.apply_transform(trimesh.transformations.rotation_matrix(np.pi / 2, [0, 0, 1]))
    elif h_axis == 2:
        mesh.apply_transform(trimesh.transformations.rotation_matrix(-np.pi / 2, [1, 0, 0]))
        
    # Đảm bảo trục Ngực nhìn thẳng vào camera (+Z)
    extents = np.ptp(mesh.vertices, axis=0)
    if extents[0] < extents[2]:
        mesh.apply_transform(trimesh.transformations.rotation_matrix(np.pi / 2, [0, 1, 0]))
        
    return mesh

def enhance_mesh_colors(mesh, source_image):
    """
    Chiếu chính xác 100% tọa độ màu sắc từ ảnh 2D gốc lên mặt trước của mô hình 3D,
    loại bỏ hoàn toàn hiện tượng bệt/loang màu và giúp chi tiết quân phục, khuôn mặt sắc nét.
    """
    try:
        import numpy as np
        img_np = np.array(source_image.convert("RGBA"))
        h, w = img_np.shape[:2]
        
        # Xác định chính xác khung hình bao quanh nhân vật trong ảnh 2D
        alpha = np.where(img_np[..., 3] > 20)
        if len(alpha[0]) == 0:
            y_min_img, y_max_img, x_min_img, x_max_img = 0, h - 1, 0, w - 1
        else:
            y_min_img, y_max_img = alpha[0].min(), alpha[0].max()
            x_min_img, x_max_img = alpha[1].min(), alpha[1].max()
            
        verts = mesh.vertices
        normals = mesh.vertex_normals
        v_colors = mesh.visual.vertex_colors.copy()
        
        x_min, x_max = verts[:, 0].min(), verts[:, 0].max()
        y_min, y_max = verts[:, 1].min(), verts[:, 1].max()
        dx = max(x_max - x_min, 1e-6)
        dy = max(y_max - y_min, 1e-6)
        
        # Tọa độ chuẩn hóa của từng đỉnh 3D
        u = (verts[:, 0] - x_min) / dx
        v = (y_max - verts[:, 1]) / dy
        
        # Ánh xạ chuẩn xác vào vị trí pixel của ảnh gốc
        px = np.clip((x_min_img + u * (x_max_img - x_min_img)).astype(int), 0, w - 1)
        py = np.clip((y_min_img + v * (y_max_img - y_min_img)).astype(int), 0, h - 1)
        
        sampled = img_np[py, px]
        
        # Trọng số theo vector pháp tuyến hướng vào camera (+Z)
        weight = np.clip(normals[:, 2], 0.0, 1.0) ** 1.5
        valid_alpha = (sampled[:, 3] > 30)
        blend_weight = (weight * valid_alpha)[:, None]
        
        new_rgb = (v_colors[:, :3] * (1.0 - blend_weight) + sampled[:, :3] * blend_weight).astype(np.uint8)
        v_colors[:, :3] = new_rgb
        mesh.visual.vertex_colors = v_colors
        return mesh
    except Exception as err:
        print(f"⚠️ Bỏ qua enhance: {err}")
        return mesh

def convert_2d_to_3d(pil_image, do_remove_bg=True, foreground_ratio=0.85, mc_resolution=320, enhance_texture=True):
    """
    Hàm nhận ảnh 2D, tách phông nền và sinh mô hình lưới 3D đứng thẳng, sắc nét (.glb)
    """
    if pil_image.mode != "RGB":
        pil_image = pil_image.convert("RGB")
        
    def fill_background(img):
        np_img = np.array(img).astype(np.float32) / 255.0
        if np_img.shape[-1] == 4:
            np_img = np_img[:, :, :3] * np_img[:, :, 3:4] + (1 - np_img[:, :, 3:4]) * 0.5
            return Image.fromarray((np_img * 255.0).astype(np.uint8))
        return img.convert("RGB")

    processed_fg = pil_image
    if do_remove_bg and rembg_session is not None:
        try:
            rembg_img = remove_background(pil_image.convert("RGB"), rembg_session)
            processed_fg = resize_foreground(rembg_img, foreground_ratio)
            input_image = fill_background(processed_fg)
        except Exception as bg_err:
            print(f"⚠️ Bỏ qua tách nền do lỗi: {bg_err}")
            input_image = pil_image.convert("RGB")
    else:
        input_image = pil_image.convert("RGB")
        
    with torch.no_grad():
        scene_codes = model_3d([input_image], device=device)
        mesh = model_3d.extract_mesh(scene_codes, has_vertex_color=True, resolution=mc_resolution)[0]
        
        # 1. Đưa mô hình đứng thẳng tuyệt đối (Head=+Y, Feet=-Y, Chest=+Z)
        mesh = orient_mesh_upright(mesh)
        
        # 2. Làm mượt vector pháp tuyến (Smooth Shading)
        mesh.fix_normals()
        
        # 3. Tăng cường độ nét màu sắc bề mặt khớp 100% với ảnh 2D
        if enhance_texture:
            mesh = enhance_mesh_colors(mesh, processed_fg)
        
    out_path = f"/content/export_3d/character_{int(time.time()*1000)}.glb"
    mesh.export(out_path)
    
    num_vertices = len(mesh.vertices)
    num_faces = len(mesh.faces)
    file_size_kb = round(os.path.getsize(out_path) / 1024, 2)
    
    mesh_stats = {
        "vertices": num_vertices,
        "faces": num_faces,
        "file_size_kb": file_size_kb,
        "file_path": out_path
    }
    return out_path, mesh_stats


In [ ]:
# =============================================================================
# BƯỚC 4: Chạy thử nghiệm Pipeline hoàn chỉnh (End-to-End) & Thu thập số liệu
# =============================================================================
# Ví dụ một prompt nhân vật Game mẫu:
sample_prompt = "cyberpunk ninja warrior with glowing katana, futuristic tactical armor, robotic cybernetic visor"

print(f"🚀 BẮT ĐẦU CHẠY THỰC NGHIỆM TẠO NHÂN VẬT: '{sample_prompt}'\n")
pipeline_start = time.time()

# 1. Sinh ảnh Concept 2D
t0 = time.time()
concept_2d = generate_2d_character(sample_prompt, steps=25)
time_2d = round(time.time() - t0, 2)
concept_2d.save("/content/concept_2d_sample.png")
print(f"✅ [Giai đoạn 1] Đã tạo xong Concept 2D: {time_2d} giây")

# 2. Chuyển Concept 2D thành Mô hình 3D
t1 = time.time()
glb_path, stats = convert_2d_to_3d(concept_2d)
time_3d = round(time.time() - t1, 2)
total_time = round(time.time() - pipeline_start, 2)
print(f"✅ [Giai đoạn 2] Đã tạo xong Mô hình 3D: {time_3d} giây")

# 3. Đo bộ nhớ GPU VRAM sử dụng
vram_used = round(torch.cuda.max_memory_allocated() / (1024**3), 2) if torch.cuda.is_available() else 0

print("\n" + "="*60)
print("📊 BẢNG KẾT QUẢ ĐO ĐẠC KHOA HỌC (CHO BÀI BÁO / BÁO CÁO):")
print("="*60)
print(f"• Thời gian tạo Concept Art 2D (Stable Diffusion) : {time_2d} s")
print(f"• Thời gian tái tạo Mô hình 3D (TripoSR)           : {time_3d} s")
print(f"• TỔNG THỜI GIAN TOÀN BỘ PIPELINE (End-to-End)   : {total_time} s")
print(f"• Số lượng đỉnh (Vertices Count)                 : {stats['vertices']:,}")
print(f"• Số lượng mặt đa giác (Faces / Triangles Count) : {stats['faces']:,}")
print(f"• Dung lượng tệp 3D (.glb)                       : {stats['file_size_kb']} KB")
print(f"• Bộ nhớ GPU tiêu thụ (Peak VRAM Usage)          : {vram_used} GB / 16.0 GB")
print(f"• Đường dẫn file 3D đầu ra                       : {glb_path}")
print("="*60)

# Hiển thị ảnh concept 2D vừa tạo
display(concept_2d)


In [ ]:
# =============================================================================
# BƯỚC 5: Giao diện Trực Quan (Gradio Web UI) - Trình diễn & Thu thập mẫu vật
# =============================================================================
import gradio as gr

def full_pipeline_interface(prompt_text, custom_image=None, mc_res=320, enhance_tex=True):
    """
    Giao diện xử lý tạo nhân vật 2D -> 3D có tùy chọn nâng cao độ nét
    """
    start = time.time()
    if custom_image is not None:
        img_input = custom_image
        stage1_info = "Sử dụng ảnh do người dùng tải lên trực tiếp."
    else:
        if not prompt_text or not prompt_text.strip():
            prompt_text = "hero knight in golden armor, front view, game character"
        img_input = generate_2d_character(prompt_text)
        stage1_info = f"Sinh ảnh 2D từ prompt: '{prompt_text}'"
        
    glb_file, mesh_info = convert_2d_to_3d(img_input, mc_resolution=int(mc_res), enhance_texture=enhance_tex)
    duration = round(time.time() - start, 2)
    
    metrics_markdown = f"""
### 📈 Số Liệu Thực Nghiệm (Scientific Metrics):
- **Thời gian xử lý tổng cộng:** `{duration} giây`
- **Độ phân giải Marching Cubes:** `{int(mc_res)}`
- **Số mặt đa giác (Faces):** `{mesh_info['faces']:,}`
- **Số đỉnh (Vertices):** `{mesh_info['vertices']:,}`
- **Kích thước file GLB:** `{mesh_info['file_size_kb']} KB`
- **Độ nét bề mặt:** `{'Đã tăng cường từ ảnh 2D gốc' if enhance_tex else 'Mặc định (Vertex Color thô)'}`
- **Ghi chú quy trình:** {stage1_info}
    """
    return img_input, glb_file, metrics_markdown

with gr.Blocks(title="AI Game Character Generator") as demo:
    gr.Markdown("""
    # 🎮 Ứng Dụng AI Trong Phát Triển Game: Tạo Hình Nhân Vật 2D $\\rightarrow$ 3D
    **Nghiên cứu ứng dụng:** Kết hợp mô hình khuếch tán (*Stable Diffusion*) và mô hình tái tạo không gian (*TripoSR Enhanced*) để tự động hóa khâu Concept-to-Asset trong sản xuất Game.
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            prompt_input = gr.Textbox(
                label="1. Nhập mô tả nhân vật (Text Prompt)",
                info="Lưu ý: Nhập mô tả ngắn gọn (< 40 từ) để tương thích CLIP 77 tokens.",
                placeholder="Ví dụ: vietnamese soldier, green military uniform, tactical vest, combat helmet, boots...",
                lines=3
            )
            image_input = gr.Image(
                label="Hoặc tải lên ảnh 2D có sẵn (Tùy chọn)",
                type="pil"
            )
            with gr.Accordion("⚙️ Tùy chọn chất lượng nâng cao (Advanced Quality)", open=True):
                mc_slider = gr.Slider(minimum=256, maximum=384, value=320, step=32, label="Độ chi tiết lưới 3D (Marching Cubes Resolution)", info="320 cho lưới dày hơn và sắc nét gấp 2.5 lần mặc định 256")
                enhance_checkbox = gr.Checkbox(label="🎨 Tăng cường độ nét chi tiết từ ảnh 2D (Texture Enhancer)", value=True, info="Chiếu trực tiếp màu chi tiết từ ảnh 2D lên mặt trước của mô hình")
            btn_run = gr.Button("🚀 Chạy Toàn Bộ Pipeline (2D + 3D)", variant="primary")
            
        with gr.Column(scale=1):
            out_image = gr.Image(label="Ảnh Concept 2D (Giai đoạn 1)")
            out_model_3d = gr.Model3D(label="Mô Hình 3D Xoay 360° (Giai đoạn 2)")
            out_metrics = gr.Markdown()
            
    btn_run.click(
        fn=full_pipeline_interface,
        inputs=[prompt_input, image_input, mc_slider, enhance_checkbox],
        outputs=[out_image, out_model_3d, out_metrics]
    )

demo.launch(theme=gr.themes.Soft(), share=True, debug=True)
